In [0]:
dbutils.widgets.text("configId", "1")
configId = dbutils.widgets.get("configId")

dbutils.widgets.text("sourceTypeId", "1")
sourceTypeId = dbutils.widgets.get("sourceTypeId")

dbutils.widgets.text("destinationContainerPath", "1")
destinationContainerPath = dbutils.widgets.get("destinationContainerPath")

dbutils.widgets.text("destinationFolderPath", "1")
destinationFolderPath = dbutils.widgets.get("destinationFolderPath")

dbutils.widgets.text("parent_run_id", "1")
pipeLineId = dbutils.widgets.get("parent_run_id")

dbutils.widgets.text("job_id", "1")
jobId = dbutils.widgets.get("job_id")


dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')

dbutils.widgets.text("ExternalLocationName", "mntprod")
ExternalLocationName = dbutils.widgets.get("ExternalLocationName")
filePathPrefix = '/dbfs/'+ExternalLocationName

dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")

In [0]:
%run ../Configurations/Init_Scripts

In [0]:
import json
import multiprocessing
import os
from multiprocessing.pool import ThreadPool
from azure.servicebus import ServiceBusClient, ServiceBusMessage, AutoLockRenewer, NEXT_AVAILABLE_SESSION,ServiceBusSubQueue
from azure.servicebus.exceptions import OperationTimeoutError
from azure.servicebus.management import MessageCountDetails
from datetime import datetime
import traceback

In [0]:
def MessageProcessing(message,receiver):
    # filePrefix = '/dbfs/mnt'
    loadType = ''
    deviceDate = ''
    deviceId = ''
    logType = ''
    destinationFolderPathDerived = ''
    destinationFileName = ''
    processedFileList = []

    fileDetails = dict()
    fileDetails['DeviceType'] = 'COM1'
    fileDetails['ConfigId'] = configId
    fileDetails['SourceTypeId'] = sourceTypeId
    fileDetails['PipelineRunId'] = pipeLineId
    fileDetails['JobId'] = jobId    

    try:

        message_json = json.loads(str(message))
        eventType = message_json.get('eventType')
        subject = message_json.get('subject')
        extension = subject.split('.')[-1]
        sourceFilePath = subject.replace('/blobServices/default/containers','').replace('/blobs','')
        # print(sourceFilePath)
        sourceFilePathList = sourceFilePath.split('/')
        sourceContainerPath = sourceFilePathList[1]
        sourceFolderPath = sourceFilePath[sourceFilePath.find('/',1)+1:sourceFilePath.rfind('/',1)]
        sourceFileName = sourceFilePathList[-1]
        logList = {'EX' : 'Exception','xGate1' : 'AgentLog', 'UE' : 'UserException', 'UL' : 'Usage', 'Assert' : 'Assert', 'xGate' : 'AgenLog',
                  'DeployStatus' : 'DeployStatus', 'Unparsed' : 'Other', 'ZInstaller' : 'InstallLogs' , 'LastContactReport' : 'COM1LastConnected',
                   'COM1' : 'Measurement'}

        if('usb' in sourceFilePathList):
            loadType = 'USB'
        else:
            loadType = 'IOT'
            
        deviceDate = sourceFileName.replace("_","")[-18:-10]
        if not(deviceDate.isnumeric()) or (len(sourceFileName) < 20):
            deviceDate = datetime.utcnow().strftime("%Y%m%d")
        if(len(sourceFilePathList)>=2) and (len(sourceFileName) > 20):
            for logListKey in logList:
                if(logListKey in sourceFileName):
                    logType = logList[logListKey]
                    break
            if(logType == ''):
                logType = 'InValid'
                    
            #logType = sourceFilePathList[2]
            if(logType == 'DeployStatus'):
                #09-30-22.12-00-30-DeployStatus
                year = '20'+sourceFileName[6:8]
                month = sourceFileName[0:2]
                day = sourceFileName[3:5]
                deviceDate = year+month+day #creating date format for folderpath
            elif(logType == 'Other'):
                deviceId = sourceFileName.split('_')[2].replace(" ","")
                deviceId = deviceId[:16]
            elif(logType == 'InstallLogs'):
                deviceId = sourceFileName.split('_')[1].replace("-","")
                deviceId = deviceId[:16]
            elif(len(sourceFileName.split('_'))>=2):
                deviceId = sourceFileName.split('_')[1].replace(" ","")
                deviceId = deviceId[:16]
            else:
                deviceId = 'InValid'
        else:
            deviceId = 'InValid'
            
            
        sourcePath = filePrefix+'/'+sourceContainerPath+'/'+sourceFolderPath+'/'+sourceFileName
        destinationFileName  = sourceFileName

        if(deviceId == 'InValid'):
            destinationFolderPathDerived = destinationFolderPath.replace('<DeviceId>','').replace('<LoadType>',loadType).replace('<LogType>','').replace('<YYYYMMDD>','')
        else:   
            destinationFolderPathDerived = destinationFolderPath.replace('<DeviceId>',deviceId).replace('<LoadType>',loadType).replace('<LogType>',logType).replace('<YYYYMMDD>',deviceDate)
            
        destinationPath = filePrefix+'/'+destinationContainerPath+'/'+destinationFolderPathDerived+'/'+destinationFileName
    except Exception as exp:
        # print(message)
        # print(str(exp))
        fileDetails['Status'] = 'Failed'
        fileDetails['ErrorMessage'] = str(exp)
        # traceback.print_exc()
#         raise
        return fileDetails        

    fileDetails['SourceFolderPath'] = '/'+sourceFolderPath+'/'
    fileDetails['SourceFileName'] = sourceFileName
    fileDetails['SourceContainerPath'] = sourceContainerPath
    fileDetails['DestinationFolderPath'] = '/'+destinationFolderPathDerived+'/'
    fileDetails['DestinationFileName'] = destinationFileName
    fileDetails['DestinationContainerPath'] = destinationContainerPath
    fileDetails['DeviceId'] = deviceId
    fileDetails['LoadType'] = loadType
    fileDetails['LogType'] = logType
    
    
    try:    
        if(eventType == "Microsoft.Storage.BlobCreated" and extension.lower() in ('csv','txt','log') and sourceContainerPath.lower() == 'logs-com1-raw'):
            
            copyFiles(sourcePath,destinationPath)
            # print("Copied:"+sourcePath,destinationPath)
            fileDetails['SourceFileSize'] = os.path.getsize(sourcePath)            
            if(deviceId == 'InValid'):
                fileDetails['PipelineStatus'] = 'InValid'
                fileDetails['ErrorMessage'] = 'Invalid log type and invalid file name'
                processedFileList.append(fileDetails)                
                logIntoIngestionLogTable(processedFileList,'LoadLogFiles_COM1')
                (receiver.dead_letter_message(message,
                                              reason=fileDetails['PipelineStatus'],
                                              error_description=fileDetails['ErrorMessage']))
                return fileDetails['ErrorMessage']
            else:
                fileDetails['PipelineStatus'] = 'Succeeded'
                fileDetails['ErrorMessage'] = ''
                processedFileList.append(fileDetails) 
                # print(processedFileList)               
                logIntoIngestionLogTable(processedFileList,'LoadLogFiles_COM1')
                receiver.complete_message(message)                
                return ''
        else:
            # print("notCopied:"+sourcePath,destinationPath)
            fileDetails['PipelineStatus'] = 'InValid'
            fileDetails['ErrorMessage'] = 'inValid File Extension not in csv,txt,log or not in logs-com1-raw container or event type is not blob created'
            processedFileList.append(fileDetails)            
            logIntoIngestionLogTable(processedFileList,'LoadLogFiles_COM1')            
            receiver.complete_message(message)
            return ''  
    except Exception as exp:
        fileDetails['PipelineStatus'] = 'Failed'
        fileDetails['ErrorMessage'] = str(exp)
        processedFileList.append(fileDetails)        
        logIntoIngestionLogTable(processedFileList,'LoadLogFiles_COM1')        
        (receiver.dead_letter_message(message,
                                      reason=fileDetails['PipelineStatus'],
                                      error_description=fileDetails['ErrorMessage']))
        # print(message)
        # print(str(exp))
        # traceback.print_exc()
#         raise
        return fileDetails['ErrorMessage']


In [0]:
def ReceiveMessageIterative(connection_string, queue_name):
    try:
        cpuCount = os.cpu_count()
        exceptionList = []
        renewer = AutoLockRenewer()        
        with ServiceBusClient.from_connection_string(connection_string) as sb_client:
            with sb_client.get_queue_receiver(queue_name=queue_name, sub_queue=ServiceBusSubQueue.DEAD_LETTER) as receiver:
                while True:
                    receivedMessages = receiver.receive_messages(max_message_count=5*cpuCount,max_wait_time=5)
                    if(len(receivedMessages)>0):
                        processedFileList = []

                        exceptionList = []
                        allMessages = []
                        for message in receivedMessages:
                            renewer.register(receiver, message, max_lock_renewal_duration=6000)
                            allMessages = allMessages + [(message, receiver)]

                        pool = ThreadPool(cpuCount)
                        returnList = pool.starmap(MessageProcessing, allMessages)
                        pool.close()
                        
                        for returnValue in returnList:
                            if returnValue:
                                exceptionList.append(returnValue)
                        
                        
                    else:
                        if(exceptionList):
                            print(exceptionList)
                            raise Exception('Exception raised while processing files and details above')
                        break
    except OperationTimeoutError:
        print("If timeout occurs during connecting to a session,It indicates that there might be no non-empty sessions remaining; exiting.This may present as a UserError in the azure portal metric.")
        raise

In [0]:
serviceBusConnectionString = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="CSServiceBus")
com1IngestionQueueName = 'agncom1ingestionframeworkqueue'
ReceiveMessageIterative(serviceBusConnectionString, com1IngestionQueueName)